# 🍍 PNPL 2026 Competition — Word Classification from Scratch

Welcome! This is the **beginner** notebook for the word-classification task at
the heart of the 2026 PNPL / LibriBrain competition.

The task in one sentence: **given a short window of brain activity (MEG)
recorded while someone listens to an audiobook, predict which word they were
hearing.**

By the end of this notebook you will have, from zero:

1. understood **what the LibriBrain100 dataset is** and how it is put together,
2. looked at a single MEG example and the event labels it comes from,
3. understood that the competition uses a **fixed 50-word vocabulary** (it is
   *not* open-vocabulary),
4. trained a small neural network that classifies MEG windows into those 50
   words,
5. evaluated it with the **official competition metric** (Top-10 Balanced
   Accuracy), and
6. written a file in the exact **submission format** the leaderboard expects.

Everything runs on the Google Colab **free** tier. On a `T4` GPU the full
notebook takes well under an hour. Set the runtime to GPU via
*Runtime → Change runtime type → T4 GPU*.

Useful links: [PNPL](https://ori.ox.ac.uk/labs/pnpl/) ·
[`pnpl` package](https://github.com/neural-processing-lab/pnpl) ·
[Competition Discord](https://discord.gg/Fqr8gJnvSh).

The companion **advanced** notebook
(`LibriBrain_Competition_Word_Classification_Advanced.ipynb`) replaces the
closed-vocabulary classifier here with a contrastive brain-to-embedding model.


## 1. Setup

Run the next cell **as is**. It installs the public `pnpl` package (our
brain-data loaders *and* the competition helpers) from GitHub, plus the handful
of extra libraries this notebook uses. It is safe to re-run — `pip` skips what
is already installed.

On Windows you may need to restart the kernel once after installation. We have
validated this notebook on Colab and Unix; other environments may vary.


In [ ]:
# --- Install dependencies and detect the runtime ---
from pathlib import Path

try:
    import google.colab  # noqa: F401  (only importable inside Colab)
    IN_COLAB = True
    BASE_PATH = Path("/content")          # the folder shown in the Colab sidebar
except ImportError:
    IN_COLAB = False
    BASE_PATH = Path(".").resolve()

# `pnpl` is pinned to the competition (`refactor`) branch. The other packages are
# standard. `-q` keeps the output short.
%pip install -q --upgrade matplotlib scikit-learn lightning tensorboard pnpl
print("Running in Colab :", IN_COLAB)
print("Base path        :", BASE_PATH)


In [ ]:
# --- Confirm the key pieces are importable and pick a device ---
import sys
from pathlib import Path

import torch
import pnpl
from pnpl.datasets import LibriBrainWord  # MEG windows aligned to word onsets

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("pnpl   :", Path(pnpl.__file__).resolve().parent)
print("Device :", DEVICE)
if DEVICE == "cpu":
    print("\n(No GPU detected. Training will work but be slower. In Colab: "
          "Runtime -> Change runtime type -> T4 GPU.)")


## 2. What is the LibriBrain100 dataset?

Before training anything, it pays to understand the data. The competition is
built on **LibriBrain100**: \~100 hours of MEG across 33 subjects, and home to
the **deepest within-subject** non-invasive neural-speech recording to date
(\~80 hours from one person). It is worth knowing how it is assembled, because it
explains the two competition tracks and where the training data comes from.

**It is a union of two Hugging Face repositories:**

| Repo | What it contains | Roughly |
| --- | --- | --- |
| [`pnpl/LibriBrain`](https://huggingface.co/datasets/pnpl/LibriBrain) | The **original** release: subject 0 listening to the first 7 Sherlock Holmes books (`Sherlock1`–`Sherlock7`). | ~50 h |
| [`pnpl/LibriBrain2`](https://huggingface.co/datasets/pnpl/LibriBrain2) | The **extension**: more subject-0 data (the rest of the Sherlock canon, TIMIT, MOCHA-TIMIT, and 30 *The Moth* podcast stories) **plus** 32 additional subjects. | ~50 h |

Together they form LibriBrain100. The design philosophy is **"deep and broad"**:

- **Deep (subject 0).** ~80 hours from a *single* participant — the entire
  canon of Sherlock Holmes audiobooks, the phonetically balanced TIMIT and
  MOCHA-TIMIT corpora, and topic-diverse podcast stories. Lots of data from one
  brain is known to drive decoding performance more than spreading the same
  budget across many brains.
- **Broad (subjects 1–32).** ~10–40 minutes *each*, all listening to the same
  two Sherlock1 sessions. This mirrors the realistic, often clinically
  constrained setting where you only get a little data from each new person.

This directly shapes the competition's two tracks:

- **Deep track** — push within-subject accuracy as far as possible on subject 0.
- **Broad track** — generalise to subjects 1–32 from limited per-subject data.

**A few facts that matter for the modelling below:**

- The recordings come from a **MEGIN TRIUX™ Neo** scanner with **306 MEG
  sensors** (102 *magnetometers* + 204 *gradiometers*) covering the whole head.
- They were recorded at 1 kHz and **downsampled to 250 Hz** during
  preprocessing, so one time sample = **4 ms**.
- For machine learning, each recording is just a 2-D array: **306 sensors ×
  time samples**, paired with a `.tsv` file of time-stamped word/phoneme labels.

> **What this notebook actually downloads.** Loading all 100 hours is overkill
> for a first model. We use only a few `Sherlock1` sessions from **subject 0** —
> i.e. a small slice of the original `pnpl/LibriBrain` part — so the download
> stays small. The `pnpl` library fetches exactly the sessions we ask for and
> nothing else. Scaling up to the full deep/broad data is a one-line change we
> point to at the end.


## 3. Load one session and look at a single example

`LibriBrainWord` does the heavy lifting: it downloads the MEG file plus its
event file, finds every word onset, and serves you a fixed-length window of MEG
around each one. We start with a **single session** so you can see the raw
ingredients before building the training pipeline.

We use a window from the **word onset** to **1.0 s after** it. At 250 Hz that is
`(1.0 - 0.0) * 250 = 250` time samples per example — deliberately the **same
window the competition holdout provides and is scored on**, so the model trains
on exactly what it will see at submission time. Here we pass `standardize=False`
so you see the data in its natural units; we let the dataset standardize for
training later.

In [ ]:
# --- Configuration shared across the notebook ---
# Run keys are (subject, session, task, run) tuples. Subject 0 is the deep
# subject; Sherlock1 sessions 1-10 use run "1". We take a small subset.
TMIN, TMAX, SFREQ = 0.0, 1.0, 250

TRAIN_RUNS = [("0", str(s), "Sherlock1", "1") for s in range(1, 5)]   # sessions 1-4
VAL_RUNS   = [("0", "5", "Sherlock1", "1")]                            # session 5
TEST_RUNS  = [("0", "6", "Sherlock1", "1")]                            # session 6

DATA_PATH = BASE_PATH / "data" / "libribrain_word"

print("Expected time samples per window:", int((TMAX - TMIN) * SFREQ))
print("Train runs:", TRAIN_RUNS)
print("Val run   :", VAL_RUNS)
print("Test run  :", TEST_RUNS)


In [ ]:
# --- Load one session and inspect a single example ---
from pnpl.datasets import LibriBrainWord

one_run = LibriBrainWord(
    data_path=str(DATA_PATH),
    include_run_keys=[TRAIN_RUNS[0]],
    tmin=TMIN,
    tmax=TMAX,
    standardize=False,     # show raw-ish values for this first look
    include_info=True,     # also return a dict with the word string, onset, etc.
    preload_files=False,   # download lazily instead of all-at-once
)

meg, label_id, info = one_run[0]
word = one_run.id_to_word[int(label_id)]

print("Examples in this session :", len(one_run))
print("MEG window shape         :", tuple(meg.shape), "  (sensors, time)")
print("Label id (within session) + which word it represents:", int(label_id), "->", repr(word))
print("Info for this example    :", info)


The MEG window is just a table of numbers: one row per sensor, one column per
time sample. You do **not** need a neuroscience background to start — for now,
treat it as a 306-channel time series. Let's visualise one.


In [ ]:
# --- Visualise the MEG window for a single word ---
import matplotlib.pyplot as plt

meg, label_id, info = one_run[0]
word = one_run.id_to_word[int(label_id)]

plt.figure(figsize=(12, 4))
plt.imshow(meg.numpy(), aspect="auto", cmap="RdBu_r",
           extent=[TMIN, TMAX, meg.shape[0], 0])
plt.axvline(0.0, color="k", lw=1, ls="--")  # word onset
plt.colorbar(label="MEG amplitude")
plt.xlabel("Time relative to word onset (s)  —  dashed line = onset")
plt.ylabel("Sensor (0-305)")
plt.title("One MEG example for the word " + repr(word))
plt.tight_layout()
plt.show()


## 4. Where do the labels come from? The event file

Each MEG recording ships with a tab-separated **event file** (`.tsv`) listing
every linguistic event (words, phonemes, …) with its time. The dataset uses the
`word` rows to know *when* each word was heard. Reading it directly demystifies
the labels — and immediately reveals a defining property of natural language:
**a handful of words are extremely common, and most words are rare** (a "long
tail").


In [ ]:
# --- Read the raw event file for one session ---
import pandas as pd

events_path = one_run.get_events_path(*TRAIN_RUNS[0])
events = pd.read_csv(events_path, sep="\t")
word_rows = events[events["kind"] == "word"].copy()

print("Event file:", events_path)
print("Total word events in this session:", len(word_rows))
word_rows.head(10)


In [ ]:
# --- The long tail: a few words dominate ---
import matplotlib.pyplot as plt
import pandas as pd

counts = (word_rows["segment"].astype(str).str.strip().str.lower()
          .value_counts())

print("Unique word types in this session:", len(counts))
print("\n20 most frequent words:")
print(counts.head(20))

counts.head(50)[::-1].plot.barh(figsize=(10, 10))
plt.title("50 most frequent words in one Sherlock1 session")
plt.xlabel("Count")
plt.tight_layout()
plt.show()


## 5. The task: a **fixed 50-word vocabulary**

The long-term goal the field is working towards is full, open-vocabulary brain-to-text. For this competition, we're simplifying the task into a closed classification problem. Performance is measured against a **fixed retrieval
vocabulary of 50 words**. Your model only ever has to choose among these 50.

This vocabulary ships *inside* the `pnpl` package, so we load it directly — no
file paths to hardcode. There are actually two 50-word lists:

- the **competition vocabulary** (the primary list — this is what the
  leaderboard scores), and
- **Moses-50** (a secondary list from a landmark invasive-BCI study, tracked for
  comparison).

We will train on the **competition vocabulary**.


In [ ]:
# --- Load the official vocabularies straight from the installed package ---
from pnpl.competition import load_vocabulary

VOCAB = load_vocabulary("primary")      # the 50 competition words, in order
MOSES = load_vocabulary("moses")        # the 50 Moses words (secondary metric)

print("Competition vocabulary size:", len(VOCAB))
print(VOCAB)
print("\nMoses-50 (secondary):")
print(MOSES)


Because the vocabulary is fixed and small, the task is a clean **50-way
classification** problem. Let's see how many of this session's examples actually
fall inside the vocabulary.

Two things to notice:

- The list is full of short, frequent **function words** (`the`, `a`, `to`,
  `it`, `i`, …) alongside common **content words** (`time`, `good`, `think`,
  `people`, `new`). This is intentional - we tried to pick a good sample of common words from the dataset.
- One entry is `it’s` (with a curly apostrophe). When we match vocabulary words
  against the dataset's word strings we will normalise apostrophes so nothing is
  silently dropped — a small but real lesson in working with text.

Because the vocabulary is fixed and small, the task is a clean **50-way
classification** problem. Let's see how many of this session's examples actually
fall inside the vocabulary.


In [ ]:
# --- Build a normalisation-robust lookup from word string -> vocab id ---
def normalize_word(w):
    # lowercase, strip whitespace, and unify the curly apostrophe (it’s -> it's)
    return str(w).strip().lower().replace("’", "'")

VOCAB_TO_ID = {normalize_word(w): i for i, w in enumerate(VOCAB)}

# How much of this one session is in-vocabulary?
in_vocab = sum(normalize_word(s[5]) in VOCAB_TO_ID for s in one_run.samples)
print("In-vocabulary examples in this session: "
      "{} / {} ({:.0%})".format(in_vocab, len(one_run.samples),
                                in_vocab / len(one_run.samples)))


## 6. Wrap the dataset to the 50-word label space

`LibriBrainWord` already returns the MEG windows we want; its built-in labels,
however, index *that session's* own word list, not the competition vocabulary.
So we add a thin wrapper that:

1. keeps only examples whose word is in the 50-word vocabulary, and
2. relabels each one with its **competition vocab id** (`0–49`).

We turn `standardize=True` back on so the dataset z-scores each MEG channel for
us — a sensible default that keeps the inputs well-behaved.


In [ ]:
# --- A thin wrapper that maps each example to the 50-word competition vocab ---
import torch
from torch.utils.data import Dataset
from pnpl.datasets import LibriBrainWord


def make_word_dataset(run_keys):
    # standardize=True z-scores each channel; clipping tames outliers.
    return LibriBrainWord(
        data_path=str(DATA_PATH),
        include_run_keys=list(run_keys),
        tmin=TMIN, tmax=TMAX,
        standardize=True,
        include_info=False,
        preload_files=False,
    )


class CompetitionVocabDataset(Dataset):
    """Filter a LibriBrainWord dataset to the 50 competition words."""

    def __init__(self, base):
        self.base = base
        # Pre-compute which base indices are in-vocabulary and their vocab id.
        self.keep = []
        for i, sample in enumerate(base.samples):
            vid = VOCAB_TO_ID.get(normalize_word(sample[5]))
            if vid is not None:
                self.keep.append((i, vid))

    def __len__(self):
        return len(self.keep)

    def __getitem__(self, idx):
        base_idx, vocab_id = self.keep[idx]
        meg = self.base[base_idx][0].float()
        return meg, torch.tensor(vocab_id, dtype=torch.long)


train_ds = CompetitionVocabDataset(make_word_dataset(TRAIN_RUNS))
val_ds   = CompetitionVocabDataset(make_word_dataset(VAL_RUNS))
test_ds  = CompetitionVocabDataset(make_word_dataset(TEST_RUNS))

print("Train / val / test in-vocab examples: "
      "{} / {} / {}".format(len(train_ds), len(val_ds), len(test_ds)))


In [ ]:
# --- How balanced is the training set across the 50 words? ---
import collections
import matplotlib.pyplot as plt

train_label_counts = collections.Counter(vid for _, vid in train_ds.keep)
counts_in_vocab_order = [train_label_counts.get(i, 0) for i in range(len(VOCAB))]

order = sorted(range(len(VOCAB)), key=lambda i: counts_in_vocab_order[i])
plt.figure(figsize=(8, 10))
plt.barh([VOCAB[i] for i in order], [counts_in_vocab_order[i] for i in order])
plt.title("Training examples per competition word")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

missing = [VOCAB[i] for i in range(len(VOCAB)) if counts_in_vocab_order[i] == 0]
print("Words with NO training examples in this subset:", missing or "none")
print("\nThe classes are highly imbalanced — this is exactly why the competition "
      "scores a *balanced* metric (next sections).")


## 7. Batch the data with `DataLoader`s

A `DataLoader` groups individual examples into mini-batches for efficient
training.


In [ ]:
# --- Wrap the datasets in DataLoaders ---
from torch.utils.data import DataLoader

BATCH_SIZE = 64
NUM_WORKERS = 2 if IN_COLAB else 0

# drop_last=True on the training loader avoids a final size-1 batch, which
# BatchNorm cannot handle in train mode.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

xb, yb = next(iter(train_loader))
print("Batch MEG shape  :", tuple(xb.shape), "  (batch, sensors, time)")
print("Batch label shape:", tuple(yb.shape))
print("First labels     :", yb[:8].tolist(),
      "->", [VOCAB[i] for i in yb[:8].tolist()])


## 8. A small 1-D convolutional model

The model is intentionally simple and easy to hack:

- stacked **`Conv1d`** layers slide over time, learning local temporal patterns
  across the 306 sensors,
- **pooling** shrinks the time axis,
- a final **linear** layer produces one score (logit) per vocabulary word.

It is not meant to be state-of-the-art — it is meant to be understandable and a
solid starting point you can improve.


In [ ]:
# --- Define the classifier ---
import torch.nn as nn


class SimpleWordCNN(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(128, 256, kernel_size=9, padding=4),
            nn.BatchNorm1d(256), nn.GELU(), nn.MaxPool1d(4),
            nn.Conv1d(256, 256, kernel_size=5, padding=2),
            nn.GELU(), nn.AdaptiveAvgPool1d(1),   # -> (batch, 256, 1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.encoder(x))


# Quick shape sanity check on one batch (no training yet).
_probe = SimpleWordCNN(n_channels=xb.shape[1], n_classes=len(VOCAB))
print("Logits for one batch:", tuple(_probe(xb).shape), "(batch, 50)")


## 9. The competition metric: Top-10 Balanced Accuracy

The leaderboard does **not** use plain accuracy. It uses **Top-10 Balanced
Accuracy** (`BAcc@10`):

- **Top-10** — a prediction counts as correct if the true word is among the
  model's **10 highest-scoring** words (out of 50). This reflects how word
  decoders are used in practice: as a shortlist a language model refines.
- **Balanced** — we average the recall *per word* and then average across the 50
  words, so every word counts equally regardless of how often it appears. This
  is essential given the imbalance we saw above.

Formally, `BAcc@10 = (1/50) · Σ_k Recall@10_k`. We also track `BAcc@1` (Top-1)
to break ties. **Random guessing scores ~20% on `BAcc@10`** for 50 words (a
random top-10 covers 10/50 of the vocabulary), so anything above ~0.20 is real
signal.


In [ ]:
# --- The official-style metric, implemented once and reused ---
import torch


def balanced_topk_accuracy(logits, labels, k, n_classes):
    # Macro-averaged Recall@k: average the per-class top-k hit rate over all
    # classes that appear in `labels`.
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices                  # (N, k)
    hit = (topk == labels.unsqueeze(1)).any(dim=1).float()  # (N,)
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.any():
            recalls.append(hit[mask].mean())
    if not recalls:
        return 0.0
    return float(torch.stack(recalls).mean())


# Sanity check: a random classifier should score ~k/n_classes on BAcc@k.
torch.manual_seed(0)
rnd_logits = torch.randn(2000, len(VOCAB))
rnd_labels = torch.randint(0, len(VOCAB), (2000,))
print("Random BAcc@10:", round(balanced_topk_accuracy(rnd_logits, rnd_labels, 10, len(VOCAB)), 3),
      "(expected ~0.20)")
print("Random BAcc@1 :", round(balanced_topk_accuracy(rnd_logits, rnd_labels, 1, len(VOCAB)), 3),
      "(expected ~0.02)")


## 10. Train with PyTorch Lightning

[Lightning](https://lightning.ai/docs/pytorch/stable/) handles the training loop
boilerplate. Our module logs the loss every epoch and computes `BAcc@10` /
`BAcc@1` on the validation set so we can watch real progress on the actual
metric.


In [ ]:
# --- Lightning module wrapping the CNN + the competition metric ---
import lightning as L
import torch
import torch.nn as nn


class LitWordClassifier(L.LightningModule):
    def __init__(self, n_channels, n_classes, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = SimpleWordCNN(n_channels, n_classes)
        self.loss_fn = nn.CrossEntropyLoss()
        self._val_logits, self._val_labels = [], []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, _):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=x.shape[0])
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        self.log("val_loss", self.loss_fn(logits, y), on_step=False,
                 on_epoch=True, prog_bar=True, batch_size=x.shape[0])
        self._val_logits.append(logits.detach().cpu())
        self._val_labels.append(y.detach().cpu())

    def on_validation_epoch_end(self):
        if not self._val_logits:
            return
        logits = torch.cat(self._val_logits)
        labels = torch.cat(self._val_labels)
        n = self.hparams.n_classes
        self.log("val_bacc10", balanced_topk_accuracy(logits, labels, 10, n), prog_bar=True)
        self.log("val_bacc1", balanced_topk_accuracy(logits, labels, 1, n), prog_bar=True)
        self._val_logits.clear()
        self._val_labels.clear()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


model = LitWordClassifier(n_channels=train_ds[0][0].shape[0], n_classes=len(VOCAB))
print(model.model)


In [ ]:
# --- Train ---
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import CSVLogger

EPOCHS = 1

csv_logger = CSVLogger(save_dir=str(BASE_PATH / "lightning_logs"), name="word_intro")

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices=1,
    logger=csv_logger,
    callbacks=[EarlyStopping(monitor="val_bacc10", mode="max", patience=5)],
    enable_checkpointing=False,
    log_every_n_steps=5,
)

trainer.fit(model, train_loader, val_loader)
print("\nLogged metrics ->", csv_logger.log_dir)


In [ ]:
# --- Plot the learning curves from the CSV log ---
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv(Path(csv_logger.log_dir) / "metrics.csv")

# One row per epoch with the last value logged for each metric.
per_epoch = metrics.groupby("epoch").last(numeric_only=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
if "train_loss" in per_epoch:
    ax1.plot(per_epoch.index, per_epoch["train_loss"], label="train")
if "val_loss" in per_epoch:
    ax1.plot(per_epoch.index, per_epoch["val_loss"], label="validation")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

if "val_bacc10" in per_epoch:
    ax2.plot(per_epoch.index, per_epoch["val_bacc10"], label="val BAcc@10")
if "val_bacc1" in per_epoch:
    ax2.plot(per_epoch.index, per_epoch["val_bacc1"], label="val BAcc@1")
ax2.axhline(0.20, color="grey", ls="--", lw=1, label="random BAcc@10")
ax2.set_title("Competition metric on validation"); ax2.set_xlabel("Epoch"); ax2.legend()
plt.tight_layout(); plt.show()


## 11. Evaluate on the held-out test session

The test session (`Sherlock1` session 6) was never seen during training. We
collect the model's scores for every test example and report the official
metrics, then look at *which* words it confuses.


In [ ]:
# --- Run the trained model over the test set ---
import torch

model.eval().to(DEVICE)
all_logits, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        all_logits.append(model(x.to(DEVICE)).cpu())
        all_labels.append(y)
all_logits = torch.cat(all_logits)
all_labels = torch.cat(all_labels)

n = len(VOCAB)
print("Test BAcc@10:", round(balanced_topk_accuracy(all_logits, all_labels, 10, n), 3),
      " (random ~0.20)")
print("Test BAcc@1 :", round(balanced_topk_accuracy(all_logits, all_labels, 1, n), 3),
      " (random ~0.02)")


In [ ]:
# --- A few individual predictions, with the model's top-5 shortlist ---
import pandas as pd

top5 = all_logits.topk(5, dim=1).indices
rows = []
for i in range(min(12, len(all_labels))):
    rows.append({
        "true_word": VOCAB[int(all_labels[i])],
        "predicted": VOCAB[int(top5[i, 0])],
        "top5_shortlist": [VOCAB[int(j)] for j in top5[i]],
    })
pd.DataFrame(rows)


In [ ]:
# --- Row-normalised confusion matrix (Top-1) over the 50 words ---
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pred1 = all_logits.argmax(dim=1).numpy()
true = all_labels.numpy()

# Restrict to words that actually occur in the test set to keep the plot legible.
present = sorted(set(true.tolist()))
labels_present = [VOCAB[i] for i in present]
cm = confusion_matrix(true, pred1, labels=present, normalize="true")

fig, ax = plt.subplots(figsize=(11, 11))
ConfusionMatrixDisplay(cm, display_labels=labels_present).plot(
    ax=ax, xticks_rotation=90, colorbar=False, cmap="Blues")
ax.set_title("Row-normalised confusion matrix (Top-1, test-set words)")
plt.tight_layout(); plt.show()


## 12. Make a competition submission

Scoring happens on a **holdout** set: we release the MEG recordings but withhold
the word labels. `pnpl` ships a loader that downloads the holdout and turns it
into the exact rows the leaderboard expects — one `(306, 250)` MEG window per
word (1 s @ 250 Hz):

```python
from pnpl.competition import LibriBrainCompetitionHoldout
holdout = LibriBrainCompetitionHoldout(track="deep")   # subject 0 — what we trained on
```

`holdout.indices` is the canonical row order — pass it straight to
`write_submission` and never reorder your predictions. (`track="deep"` is
subject 0, this notebook; `track="broad"` is subjects 1–39.)

The submission format is one row per example:

```
index, <50 competition-vocab probabilities>, moses_<word> probabilities...
```

- the first 50 columns are a distribution over the **competition vocabulary**, in
  `VOCAB` order — this is what gets scored on `BAcc@10`,
- an optional 50 `moses_…` columns carry the Moses-50 distribution (secondary).

Two things must match training:

1. **Preprocessing.** The holdout MEG is *raw*, so we z-score each channel with the
   **training** statistics and clip — exactly what `standardize=True` did during
   training.
2. **Window.** The holdout serves a fixed `[onset, onset+1 s]` window — the same
   1 s window `TMIN, TMAX = 0.0, 1.0` gives you during training. (The loader always
   yields this window, so keep `TMIN`/`TMAX` within `[0, 1]` s if you experiment.)

In [ ]:
# --- Build a real submission from the competition holdout recordings ---
import torch
import pandas as pd
from pnpl.competition import LibriBrainCompetitionHoldout, write_submission

# One (306, 250) MEG window per holdout word, in canonical leaderboard order.
holdout = LibriBrainCompetitionHoldout(track="deep")     # subject 0
print("Holdout rows:", len(holdout), holdout.counts())

# Preprocess each window EXACTLY as in training: z-score with the *training*
# channel statistics, then clip. (The holdout MEG is raw — skip this and the
# inputs are ~1e-12 and the model output is meaningless.)
means = torch.tensor(train_ds.base.channel_means, dtype=torch.float32)[None, :, None]
stds  = torch.tensor(train_ds.base.channel_stds,  dtype=torch.float32)[None, :, None]

model.eval().to(DEVICE)
probs = []
with torch.no_grad():
    for meg, _meta in holdout.iter_windows(batch_size=BATCH_SIZE):   # meg: (B, 306, 250)
        x = (torch.from_numpy(meg) - means) / stds
        x = x.clamp(-10, 10)
        logits = model(x.to(DEVICE))
        probs.append(torch.softmax(logits, dim=1).cpu())
probs = torch.cat(probs).numpy()

submission_path = write_submission(
    path=str(BASE_PATH / "submission_deep.csv"),
    indices=holdout.indices,        # canonical row order — do not reorder
    primary_probs=probs,            # (N, 50) over the competition vocabulary
    # secondary_probs=...,          # optional: a (N, 50) Moses-50 distribution
)
print("Wrote submission to:", submission_path)

preview = pd.read_csv(submission_path)
print("Shape:", preview.shape, " (rows, 1 index + 50 word columns)")
preview.iloc[:5, :6]

That's a real, uploadable **Deep-track** submission (subject 0). To put it on the
leaderboard, upload with the Kaggle helper (see
[`examples/submit_pnpl2026.ipynb`](https://github.com/neural-processing-lab/pnpl/blob/refactor/examples/submit_pnpl2026.ipynb)):


In [ ]:
# Ensure you provide a Kaggle API token to authenticate
%env KAGGLE_API_TOKEN=KGAT_xyz

from pnpl.competition import submit_to_kaggle

# Pick which track to submit to here
# submit_to_kaggle(submission_path, competition="pnpl-competition-2026-broad", message="Hello from the tutorial!")
submit_to_kaggle(submission_path, competition="pnpl-competition-2026-deep", message="Hello from the tutorial!")

> **Broad track.** `LibriBrainCompetitionHoldout(track="broad")` covers subjects
> 1–39. Those subjects need cross-subject training (`LibriBrain100Word`) and
> per-subject standardisation — reuse each subject's own channel statistics rather
> than subject 0's.

## 13. Where to go next

You now have a complete pipeline: load MEG → 50-way classifier → official metric
→ submission file. The highest-leverage things to try:

1. **More data.** Add more `Sherlock1` sessions to `TRAIN_RUNS`, or switch to
   `LibriBrainWord(partition="train")` for all of subject 0's standard training
   split. To use the *full* deep + broad LibriBrain100, use `LibriBrain100Word`
   with its `subjects` / `corpus` selectors:

   ```python
   from pnpl.datasets import LibriBrain100Word
   ds = LibriBrain100Word(
       data_path="./data/LibriBrain100",
       partition="train",
       subjects="deep",      # "broad", "all", 0, [1, 2, 3], range(1, 33)
       corpus="sherlock",    # "timit", "mocha", "podcasts", "all"
   )
   ```

2. **A longer or shorter window** (`TMIN` / `TMAX`) around word onset.
3. **A stronger architecture** — deeper CNNs, a transformer, or a temporal model.
4. **Handle class imbalance** — e.g. class-weighted loss or resampling.

Happy decoding! Questions? Join the
[Competition Discord](https://discord.gg/Fqr8gJnvSh).
